# Music Source Separation - Data Exploration

This notebook explores the audio data and visualizes spectrograms for music source separation.

In [1]:
import sys
sys.path.append('..')

import torch
import torchaudio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import IPython.display as ipd

from src.utils import load_audio, stft, istft, magnitude_phase, load_config
from src.dataset import MusicSeparationDataset

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Configuration

In [2]:
config = load_config('../config.yaml')
print("Configuration:")
print(f"Sample Rate: {config['data']['sample_rate']} Hz")
print(f"N_FFT: {config['data']['n_fft']}")
print(f"Hop Length: {config['data']['hop_length']}")
print(f"Segment Length: {config['data']['segment_length']} seconds")
print(f"Stems: {config['data']['stems']}")

Configuration:
Sample Rate: 44100 Hz
N_FFT: 2048
Hop Length: 512
Segment Length: 6.0 seconds
Stems: ['vocals', 'drums', 'bass', 'other']


## 2. Check Data Directory Structure

In [ ]:
raw_dir = Path(config['data']['raw_dir'])
stems_dir = Path(config['data']['stems_dir'])

print(f"Raw directory: {raw_dir}")
print(f"Exists: {raw_dir.exists()}")

if raw_dir.exists():
    raw_files = list(raw_dir.glob('*.wav'))
    print(f"\nFound {len(raw_files)} audio files:")
    for f in raw_files[:5]:  # Show first 5
        print(f"  - {f.name}")
else:
    print("\n⚠️ Raw directory doesn't exist. Please create it and add audio files.")
    print(f"   mkdir -p {raw_dir}")

print(f"\nStems directory: {stems_dir}")
print(f"Exists: {stems_dir.exists()}")

if stems_dir.exists():
    stem_folders = [d for d in stems_dir.iterdir() if d.is_dir()]
    print(f"\nFound {len(stem_folders)} stem folders:")
    for folder in stem_folders[:3]:  # Show first 3
        stems = list(folder.glob('*.wav'))
        print(f"  - {folder.name}: {len(stems)} stems")
else:
    print("\n⚠️ Stems directory doesn't exist. Please create it.")
    print(f"   mkdir -p {stems_dir}")

## 3. Load and Visualize Audio

Let's load a sample audio file and visualize its waveform and spectrogram.

In [ ]:
# Load a sample audio file
sample_file = raw_files[0] if raw_dir.exists() and len(raw_files) > 0 else None

if sample_file:
    print(f"Loading: {sample_file.name}")
    audio = load_audio(str(sample_file), sample_rate=config['data']['sample_rate'])
    
    print(f"Audio shape: {audio.shape}")
    print(f"Duration: {audio.shape[1] / config['data']['sample_rate']:.2f} seconds")
    print(f"Channels: {audio.shape[0]}")
    
    # Play audio
    print("\nAudio Player:")
    ipd.display(ipd.Audio(audio.numpy(), rate=config['data']['sample_rate']))
else:
    print("⚠️ No audio files found. Please add audio files to the raw directory.")

### 3.1 Waveform Visualization

In [ ]:
if sample_file:
    # Plot waveform
    fig, axes = plt.subplots(2, 1, figsize=(15, 6))
    
    time_axis = np.arange(audio.shape[1]) / config['data']['sample_rate']
    
    # Left channel
    axes[0].plot(time_axis, audio[0].numpy(), linewidth=0.5)
    axes[0].set_title('Left Channel Waveform')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    
    # Right channel
    axes[1].plot(time_axis, audio[1].numpy(), linewidth=0.5, color='orange')
    axes[1].set_title('Right Channel Waveform')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Amplitude')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 3.2 Spectrogram Visualization

In [ ]:
if sample_file:
    # Compute STFT
    spec = stft(
        audio,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    # Get magnitude and phase
    magnitude, phase = magnitude_phase(spec)
    
    print(f"Spectrogram shape: {magnitude.shape}")
    print(f"Frequency bins: {magnitude.shape[1]}")
    print(f"Time frames: {magnitude.shape[2]}")

In [ ]:
if sample_file:
    # Plot magnitude spectrogram (in dB)
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    
    # Convert to dB scale
    mag_db_left = 20 * torch.log10(magnitude[0] + 1e-8)
    mag_db_right = 20 * torch.log10(magnitude[1] + 1e-8)
    
    # Left channel
    im1 = axes[0].imshow(
        mag_db_left.numpy(),
        aspect='auto',
        origin='lower',
        cmap='viridis',
        vmin=-80,
        vmax=0
    )
    axes[0].set_title('Left Channel Spectrogram (dB)')
    axes[0].set_xlabel('Time Frames')
    axes[0].set_ylabel('Frequency Bins')
    plt.colorbar(im1, ax=axes[0], label='Magnitude (dB)')
    
    # Right channel
    im2 = axes[1].imshow(
        mag_db_right.numpy(),
        aspect='auto',
        origin='lower',
        cmap='viridis',
        vmin=-80,
        vmax=0
    )
    axes[1].set_title('Right Channel Spectrogram (dB)')
    axes[1].set_xlabel('Time Frames')
    axes[1].set_ylabel('Frequency Bins')
    plt.colorbar(im2, ax=axes[1], label='Magnitude (dB)')
    
    plt.tight_layout()
    plt.show()

## 4. Visualize Individual Stems

In [ ]:
if sample_file and stems_dir.exists():
    song_name = sample_file.stem
    song_stems_dir = stems_dir / song_name
    
    if song_stems_dir.exists():
        print(f"Loading stems for: {song_name}")
        
        stems_audio = {}
        for stem_name in config['data']['stems']:
            stem_file = song_stems_dir / f"{stem_name}.wav"
            if stem_file.exists():
                stems_audio[stem_name] = load_audio(
                    str(stem_file),
                    sample_rate=config['data']['sample_rate']
                )
                print(f"  ✓ Loaded {stem_name}: {stems_audio[stem_name].shape}")
            else:
                print(f"  ✗ Missing {stem_name}")
    else:
        print(f"⚠️ Stem directory not found: {song_stems_dir}")
        stems_audio = {}

In [ ]:
if stems_audio:
    # Plot spectrograms for all stems
    num_stems = len(stems_audio)
    fig, axes = plt.subplots(num_stems, 1, figsize=(15, 4 * num_stems))
    
    if num_stems == 1:
        axes = [axes]
    
    for idx, (stem_name, stem_audio) in enumerate(stems_audio.items()):
        # Compute spectrogram
        stem_spec = stft(
            stem_audio,
            n_fft=config['data']['n_fft'],
            hop_length=config['data']['hop_length'],
            win_length=config['data']['win_length']
        )
        stem_mag, _ = magnitude_phase(stem_spec)
        
        # Convert to dB (use left channel)
        stem_mag_db = 20 * torch.log10(stem_mag[0] + 1e-8)
        
        # Plot
        im = axes[idx].imshow(
            stem_mag_db.numpy(),
            aspect='auto',
            origin='lower',
            cmap='viridis',
            vmin=-80,
            vmax=0
        )
        axes[idx].set_title(f'{stem_name.capitalize()} Spectrogram (dB)')
        axes[idx].set_xlabel('Time Frames')
        axes[idx].set_ylabel('Frequency Bins')
        plt.colorbar(im, ax=axes[idx], label='Magnitude (dB)')
    
    plt.tight_layout()
    plt.show()

### 4.1 Listen to Individual Stems

In [ ]:
if stems_audio:
    for stem_name, stem_audio in stems_audio.items():
        print(f"\n{stem_name.capitalize()}:")
        ipd.display(ipd.Audio(stem_audio.numpy(), rate=config['data']['sample_rate']))

## 5. Test Dataset Class

In [ ]:
try:
    dataset = MusicSeparationDataset(
        raw_dir=config['data']['raw_dir'],
        stems_dir=config['data']['stems_dir'],
        stems=config['data']['stems'],
        sample_rate=config['data']['sample_rate'],
        segment_length=config['data']['segment_length'],
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length'],
        train=True
    )
    
    print(f"Dataset size: {len(dataset)}")
    
    # Get a sample
    if len(dataset) > 0:
        sample = dataset[0]
        print("\nSample batch contents:")
        print(f"  mix_magnitude shape: {sample['mix_magnitude'].shape}")
        print(f"  mix_phase shape: {sample['mix_phase'].shape}")
        print(f"  mix_audio shape: {sample['mix_audio'].shape}")
        print(f"  song_name: {sample['song_name']}")
        print("\n  Stem magnitudes:")
        for stem_name, stem_mag in sample['stem_magnitudes'].items():
            print(f"    {stem_name}: {stem_mag.shape}")
except Exception as e:
    print(f"⚠️ Error creating dataset: {e}")
    print("\nMake sure you have:")
    print("1. Audio files in data/raw/")
    print("2. Corresponding stems in data/stems/song_name/")

## 6. Test STFT/iSTFT Reconstruction

In [ ]:
if sample_file:
    # Take a short segment
    segment_samples = int(5.0 * config['data']['sample_rate'])  # 5 seconds
    audio_segment = audio[:, :segment_samples]
    
    # STFT
    spec = stft(
        audio_segment,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    # iSTFT
    reconstructed = istft(
        spec,
        n_fft=config['data']['n_fft'],
        hop_length=config['data']['hop_length'],
        win_length=config['data']['win_length']
    )
    
    # Compute reconstruction error
    min_len = min(audio_segment.shape[1], reconstructed.shape[1])
    error = torch.mean((audio_segment[:, :min_len] - reconstructed[:, :min_len]) ** 2)
    
    print(f"Original shape: {audio_segment.shape}")
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Mean Squared Error: {error.item():.6f}")
    
    # Play both
    print("\nOriginal:")
    ipd.display(ipd.Audio(audio_segment.numpy(), rate=config['data']['sample_rate']))
    print("\nReconstructed:")
    ipd.display(ipd.Audio(reconstructed.numpy(), rate=config['data']['sample_rate']))

## 7. Data Statistics

In [ ]:
if sample_file:
    print("Audio Statistics:")
    print(f"  Mean: {audio.mean().item():.6f}")
    print(f"  Std: {audio.std().item():.6f}")
    print(f"  Min: {audio.min().item():.6f}")
    print(f"  Max: {audio.max().item():.6f}")
    
    print("\nSpectrogram Statistics (Magnitude):")
    print(f"  Mean: {magnitude.mean().item():.6f}")
    print(f"  Std: {magnitude.std().item():.6f}")
    print(f"  Min: {magnitude.min().item():.6f}")
    print(f"  Max: {magnitude.max().item():.6f}")

## 8. Next Steps

Now that you've explored your data:

1. **Ensure data is properly organized:**
   - Mixed audio in `data/raw/`
   - Stems in `data/stems/song_name/`

2. **Adjust config.yaml if needed:**
   - Sample rate
   - STFT parameters
   - Segment length

3. **Start training:**
   ```bash
   python src/train.py --config config.yaml
   ```

4. **Monitor with TensorBoard:**
   ```bash
   tensorboard --logdir logs
   ```